# Deep Ensemble OOD Detection

Train 10 EfficientNet-B0 models on UC Merced (different seeds). At test time, use the
**variance across models** as an OOD score:

- UC Merced test samples → all 10 models agree → **low** variance
- FIDS30 samples → models disagree (different basins extrapolate differently) → **high** variance

In [ ]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torchvision import datasets
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

NUM_CLASSES = 21
N_MODELS = 10
EPOCHS = 5
BATCH_SIZE = 64
LR = 1e-4
SEED = 42
SAVE_DIR = Path("ensemble_10")
SAVE_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Train 10 Models on UC Merced

Skips any model whose checkpoint already exists, so you can resume if interrupted.

In [ ]:
def build_model():
    return timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)

cfg = timm.data.resolve_model_data_config(build_model())
train_transform = timm.data.create_transform(**cfg, is_training=True)
val_transform = timm.data.create_transform(**cfg, is_training=False)

train_ds = datasets.ImageFolder("PrepData/UC_Merced/Training", transform=train_transform)
val_ds = datasets.ImageFolder("PrepData/UC_Merced/Validation", transform=val_transform)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

criterion = nn.CrossEntropyLoss()

for k in range(N_MODELS):
    path = SAVE_DIR / f"model_{k}.pth"
    if path.exists():
        print(f"Model {k}: already exists, skipping")
        continue

    print(f"\n=== Training Model {k+1}/{N_MODELS} (seed={SEED+k}) ===")
    torch.manual_seed(SEED + k)
    model = build_model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    g = torch.Generator().manual_seed(SEED + k)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=True, generator=g)

    for epoch in range(EPOCHS):
        model.train()
        for images, t in tqdm(train_loader, desc=f"  Epoch {epoch+1}/{EPOCHS}", leave=False):
            images, t = images.to(device), t.to(device)
            loss = criterion(model(images), t)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for images, t in val_loader:
                images, t = images.to(device), t.to(device)
                correct += (model(images).argmax(1) == t).sum().item()
        print(f"  Epoch {epoch+1}: Val Acc={100*correct/len(val_ds):.2f}%")

    torch.save(model.state_dict(), path)
    print(f"  Saved to {path}")

## 2. Inference: Collect 10 Softmax Outputs per Sample

Run each model (dropout off) on UC Merced test + FIDS30. Stack into (N_MODELS, N_samples, N_classes).

In [ ]:
uc_test_ds = datasets.ImageFolder("PrepData/UC_Merced/Test", transform=val_transform)
fids_ds = datasets.ImageFolder("PrepData/FIDS30", transform=val_transform)

uc_loader = DataLoader(uc_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
fids_loader = DataLoader(fids_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

def run_ensemble(loader):
    passes = []
    for k in tqdm(range(N_MODELS), desc="Models"):
        model = build_model().to(device)
        model.load_state_dict(torch.load(SAVE_DIR / f"model_{k}.pth", map_location=device))
        model.eval()
        probs = []
        with torch.no_grad():
            for images, _ in loader:
                images = images.to(device)
                probs.append(F.softmax(model(images), dim=1).cpu())
        passes.append(torch.cat(probs))
    return torch.stack(passes)  # (N_MODELS, N, C)

uc_probs = run_ensemble(uc_loader)
fids_probs = run_ensemble(fids_loader)

print(f"UC Merced probs shape: {uc_probs.shape}")
print(f"FIDS30 probs shape:    {fids_probs.shape}")

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## 3. Variance as OOD Score

In [ ]:
uc_variance = uc_probs.var(dim=0).mean(dim=1).numpy()
fids_variance = fids_probs.var(dim=0).mean(dim=1).numpy()

print(f"UC Merced: mean variance = {uc_variance.mean():.4f}")
print(f"FIDS30:    mean variance = {fids_variance.mean():.4f}")
print(f"Ratio:     {fids_variance.mean() / uc_variance.mean():.2f}x")

#Trained from scratch
#UC Merced: mean variance = 0.0013
#FIDS30:    mean variance = 0.0131
#Ratio:     10.32x

## 4. Histogram + Threshold Analysis

In [ ]:
threshold = np.percentile(uc_variance, 95)
false_alarm = (uc_variance > threshold).mean()
detection = (fids_variance > threshold).mean()

print(f"Threshold (95th pct of UC Merced variance): {threshold:.4f}")
print(f"  False alarm rate (UC Merced > threshold):  {false_alarm:.1%}")
print(f"  Detection rate   (FIDS30 > threshold):     {detection:.1%}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(uc_variance, bins=40, alpha=0.7, label=f"UC Merced (n={len(uc_variance)})", color='steelblue')
ax.hist(fids_variance, bins=40, alpha=0.7, label=f"FIDS30 (n={len(fids_variance)})", color='tomato')
ax.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f"Threshold = {threshold:.3f}")
ax.set_xlabel("Normalized variance across ensemble")
ax.set_ylabel("Count")
ax.set_title(f"Deep Ensemble OOD: {detection:.0%} detection, {false_alarm:.0%} false alarm")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()